# ScholarAgent error analysis (Phase 8)

Load a saved evaluation run and inspect failures by system and question type.
Default paths assume you ran:

```bash
uv run scholar-agent evaluate --max-questions 50 --embedding-backend hash
```

In [ ]:
from pathlib import Path
import json
import pandas as pd

OUT = Path("outputs/evaluation")
results_path = OUT / "results.json"
fail_path = OUT / "failures.json"
pq_path = OUT / "per_question_metrics.csv"

report = json.loads(results_path.read_text()) if results_path.is_file() else {}
failures = json.loads(fail_path.read_text()) if fail_path.is_file() else []
print("run_id:", report.get("run_id"))
print("fingerprint:", report.get("frozen_split_fingerprint"))
print("n failures:", len(failures))
if pq_path.is_file():
    df = pd.read_csv(pq_path)
    display(df.groupby(["system", "question_type"])[["recall_at_k_paper", "token_f1", "latency_ms"]].mean())
else:
    df = None
    print("per_question_metrics.csv missing — run evaluate first")

## Manual review checklist (≥5 cases)

1. Dense miss on keyword/acronym query
2. Graph noise on simple factual query
3. Unanswerable refusal quality
4. Comparison missing one paper side
5. Weak citation / page provenance

See `docs/failure_analysis.md` for the written narratives.

In [ ]:
from collections import Counter
print(Counter((f.get("system"), f.get("question_type")) for f in failures).most_common(20))
for f in failures[:10]:
    print("-" * 60)
    print(f.get("system"), f.get("question_id"), f.get("question_type"))
    print(f.get("question"))
    print("reason:", f.get("reason"))
    print("preview:", (f.get("answer_preview") or "")[:200])